# Mamba-Attention Recursive Reasoning Hybrid Framework

A modular PyTorch library combining **Structured State Duality (SSD/Mamba-2)** and **attention mechanisms**
for recursive reasoning over symbolic tasks (Sudoku, Maze, Dijkstra, GSM8K).

**Architecture Paradigm: Thinking Fast & Slow**
- **Planning (Slow):** Bidirectional attention-based latent planning loop that recursively refines memory
  and task-constraint representations (meta-tokens) over variable compute steps.
- **Generation (Fast):** Autoregressive Mamba-2 generator decodes the final solution sequence.

---
**Quickstart:** `uv sync` → then run cells below.

## 1. Setup & Environment

In [ ]:
import json
import os
import random
from typing import List

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

device = torch.device(
    "xpu"
    if hasattr(torch, "xpu") and torch.xpu.is_available()
    else ("cuda" if torch.cuda.is_available() else "cpu")
)
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Training is opt-in so Run All remains a quick smoke test.
RUN_TRAINING = False

## 2. Configuration

The `MambaHybridConfig` dataclass controls all model hyperparameters:

In [ ]:
from mamba_hybrid.config import MambaHybridConfig

config = MambaHybridConfig(
    d_model=128,  # Model dimension
    n_meta=32,  # Number of latent meta-tokens (planning state)
    l_ans=81,  # Answer sequence length
    n_steps=4,  # Latent steps per cycle
    t_cycles=3,  # Total cycles (T-1 warmup, 1 supervised)
    use_cuda_kernels=False,  # Pure PyTorch fallback for CPU/GPU compat
)
print(config)

## 3. Architecture Deep-Dive

### 3a. Core Hybrid Block (`MambaAttentionHybridBlock`)

Each hybrid block runs **Attention** and **Mamba-2 SSD** in parallel, then fuses them with RMSNorm + learned beta scaling.

In [ ]:
from mamba_hybrid.operators import MambaAttentionHybridBlock

block = MambaAttentionHybridBlock(config)
dummy = torch.randn(2, 16, config.d_model)
out = block(dummy, causal=False)
print(f"Input shape:  {dummy.shape}")
print(f"Output shape: {out.shape}")

### 3b. Prefix-Causal Attention

When `causal=True`, the attention mask allows bidirectional communication among meta-tokens,
while subsequent (generated) tokens attend causally. When `causal=False`, all tokens attend
bidirectionally (used during planning).

In [ ]:
from mamba_hybrid.attention import PrefixCausalAttention

attn = PrefixCausalAttention(config)
B, H, L, D_head = 2, 8, 16, config.d_model // 8
q = torch.randn(B, H, L, D_head)
k = torch.randn(B, H, L, D_head)
v = torch.randn(B, H, L, D_head)

out_non_causal = attn(q, k, v, causal=False)
out_causal = attn(q, k, v, causal=True)
print(f"Bidirectional: {out_non_causal.shape}")
print(f"Prefix-causal: {out_causal.shape}")

### 3c. Mamba-2 SSD Scan

Pure PyTorch sequential scan serving as CPU/GPU-compatible fallback.
Optionally uses GPU-accelerated Triton kernels from `mamba-ssm` when available.

In [ ]:
from mamba_hybrid.ssm import Mamba2SSDScan

ssm = Mamba2SSDScan(
    config.d_model, expansion=2, num_heads=8, d_state=16, use_cuda_kernels=False
)
B, L = 2, 16
x = torch.randn(B, L, config.d_model * 2)
gate = torch.randn(B, L, config.d_model * 2)
h_in = torch.randn(B, L, 8 * 16)
h_out = torch.randn(B, L, 8 * 16)
delta = torch.randn(B, L, 8)
y = ssm(x, gate, h_in, h_out, delta)
print(f"SSM output shape: {y.shape}")

### 3d. Planning Loop

The `PlanningLoop` runs recurrent latent steps to update `z` (meta-tokens),
then calls cross-attention `AnswerUpdateBlock` to update `y` (answer state) at the end of each cycle.

**Full-recursion BPTT:** The `warmup` argument is retained for API compatibility, but every cycle
keeps its computational graph. Gradients therefore reach the learned initial states across all cycles.

In [ ]:
from mamba_hybrid.planning import PlanningLoop

planner = PlanningLoop(config)
B = 2
x_raw = torch.randn(B, 81, config.d_model)
z = torch.randn(B, config.n_meta, config.d_model)
y = torch.randn(B, config.l_ans, config.d_model)

# Warmup pass (no gradients stored)
z_out, y_out = planner(x_raw, z, y, warmup=True)
print(f"Warmup - z: {z_out.shape}, y: {y_out.shape}")

# Gradient-enabled pass
z_out2, y_out2 = planner(x_raw, z, y, warmup=False)
print(f"Grad    - z: {z_out2.shape}, y: {y_out2.shape}")

### 3e. ACT Halting Module

Two halting heads are available:
- **BCE Mode:** Binary classifier predicting halt probability at each step
- **Q-learning Mode:** Q-values for continue vs. halt actions

Both use detached global average pooling to prevent gradients from flowing back into representations.

In [ ]:
from mamba_hybrid.halting import ACTHaltingModule

q_head = ACTHaltingModule(config)
z_dummy = torch.randn(B, config.n_meta, config.d_model)
y_dummy = torch.randn(B, config.l_ans, config.d_model)

bce_prob = q_head(z_dummy, y_dummy)
q_vals = q_head.get_q_values(z_dummy, y_dummy)
print(
    f"BCE halt prob:  {bce_prob.shape}  (values: {[round(value, 3) for value in bce_prob.detach().tolist()]})"
)
print(f"Q-values:       {q_vals.shape}  (continue, halt)")

### 3f. Full Model: `MambaAttentionHybrid`

Assembles all components:
1. Initializes `z` from learned `M_meta` parameter and `y` via pooled input projection
2. Runs T-1 warmup cycles (no gradients)
3. Runs final supervised cycle with `n_steps` latent updates + ACT halting
4. Final answer update via cross-attention

In [ ]:
from mamba_hybrid.model import MambaAttentionHybrid

model = MambaAttentionHybrid(config)
X_raw = torch.randn(B, 81, config.d_model)
y_final, bce_probs = model(X_raw)
print(f"Final answer:     {y_final.shape}")
print(f"BCE probs length: {len(bce_probs)}")
print(f"BCE probs[0]:     {bce_probs[0].shape}")

## 4. Data Loading

The framework supports four reasoning tasks with distinct data formats.

In [ ]:
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

# Check available datasets
available = []
if os.path.exists(f"{DATA_DIR}/sudoku.jsonl"):
    available.append("sudoku (9x9)")
if os.path.exists(f"{DATA_DIR}/maze_dryrun.pt"):
    available.append("maze 10x10")
if os.path.exists(f"{DATA_DIR}/maze_hard.pt"):
    available.append("maze 30x30")
if os.path.exists(f"{DATA_DIR}/dijkstra.pt"):
    available.append("dijkstra (20-node)")
if os.path.exists(f"{DATA_DIR}/gsm8k_train.jsonl"):
    available.append("gsm8k (math)")
print(
    f"Available datasets: {available if available else 'NONE - run download_all_datasets first'}"
)

### 4a. Sudoku Dataset

9x9 boards flattened to 81 tokens. Supports data augmentation via digit permutation,
transposition, and flips.

In [ ]:
if os.path.exists(f"{DATA_DIR}/sudoku.jsonl"):
    from scripts.train_sudoku import SudokuDataset

    all_samples = []
    with open(f"{DATA_DIR}/sudoku.jsonl") as f:
        for line in f:
            all_samples.append(json.loads(line))
            if len(all_samples) >= 100:
                break

    ds = SudokuDataset(all_samples, augment=True)
    puzzle, solution = ds[0]
    print(f"Puzzle shape:   {puzzle.shape}")
    print(f"Solution shape: {solution.shape}")
    print(f"Puzzle[:27]:    {puzzle[:27].tolist()}")
    print(f"Solution[:27]:  {solution[:27].tolist()}")

### 4b. Maze Dataset

Grid (10x10 or 30x30) flattened to 100/900 tokens. Path tokens are flat indices `r * size + c`.
Uses 2D row/column positional embeddings.

In [ ]:
if os.path.exists(f"{DATA_DIR}/maze_dryrun.pt"):
    from scripts.train_maze import MazeDataset, MazeReasoningModel

    raw = torch.load(f"{DATA_DIR}/maze_dryrun.pt")
    max_path_len = max(len(sample["path"]) for sample in raw)
    ds = MazeDataset(f"{DATA_DIR}/maze_dryrun.pt", size=10, max_path_len=max_path_len)
    grid_flat, path_tokens = ds[0]
    print(f"Grid flat shape: {grid_flat.shape}")
    print(f"Path tokens:     {path_tokens.tolist()}")
    print(f"Grid samples:    {len(raw)}")

### 4c. Dijkstra Dataset

20-node weighted adjacency graphs. Input: continuous features `[num_nodes, d_model]`.
Target: parent indices (shortest path tree).

In [ ]:
if os.path.exists(f"{DATA_DIR}/dijkstra.pt"):
    from scripts.train_dijkstra import DijkstraDataset, DijkstraReasoningModel

    samples = torch.load(f"{DATA_DIR}/dijkstra.pt")
    ds = DijkstraDataset(samples, augment=True, num_nodes=20, d_model=128)
    feats, parents = ds[0]
    print(f"Features shape: {feats.shape}")
    print(f"Parents shape:  {parents.shape}")
    print(f"Parents:        {parents.tolist()}")

### 4d. Multi-Task Dataset

Combines Maze, Sudoku, Dijkstra, and GSM8K with task-prefixed input strings.
Each sample includes a `task_name` for routing through MoE layers and task-specific heads.

In [ ]:
if os.path.exists(f"{DATA_DIR}/sudoku.jsonl"):
    from scripts.train_multitask import (
        MultiTaskDataset,
        UnifiedReasoningLLM,
    )

    mt_ds = MultiTaskDataset(
        DATA_DIR, max_seq_len=128, l_ans=128, max_samples_per_task=10, required_tasks=()
    )
    print(f"Total multi-task samples: {len(mt_ds)}")
    if len(mt_ds) > 0:
        inp_ids, tgt_ids, task_name = mt_ds[0]
        inp_text = "".join(chr(t) for t in inp_ids.tolist() if t != 0)
        tgt_text = "".join(chr(t) for t in tgt_ids.tolist() if t != 0)
        print(f"Task:    {task_name}")
        print(f"Input:   {inp_text[:80]}...")
        print(f"Target:  {tgt_text[:80]}...")

## 5. Training

### 5a. Sudoku Solver

Trains the model to solve 9x9 Sudoku boards. Uses digit augmentation for generalization.

In [ ]:
def train_sudoku(config_override: dict | None = None) -> nn.Module | None:
    data_path = f"{DATA_DIR}/sudoku.jsonl"
    if not os.path.exists(data_path):
        print("Sudoku dataset not found. Skipping.")
        return None

    cfg = MambaHybridConfig(
        d_model=128, n_meta=32, l_ans=81, n_steps=4, t_cycles=3, vocab_size=10
    )
    if config_override:
        for k, v in config_override.items():
            setattr(cfg, k, v)

    from scripts.train_sudoku import SudokuDataset, SudokuReasoningModel
    from mamba_hybrid.loss import compute_bce_joint_loss

    all_samples = []
    with open(data_path) as f:
        for line in f:
            all_samples.append(json.loads(line))
            if len(all_samples) >= 5000:
                break
    random.seed(42)
    random.shuffle(all_samples)
    train_size = int(0.8 * len(all_samples))
    train_loader = DataLoader(
        SudokuDataset(all_samples[:train_size], augment=True),
        batch_size=32,
        shuffle=True,
    )
    val_loader = DataLoader(
        SudokuDataset(all_samples[train_size:], augment=False),
        batch_size=32,
        shuffle=False,
    )

    model = SudokuReasoningModel(cfg, vocab_size=10).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

    for epoch in range(1, 4):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0
        for inp, tgt in train_loader:
            inp, tgt = inp.to(device), tgt.to(device)
            opt.zero_grad()
            logits, bce_probs = model(inp)
            preds = logits.argmax(dim=-1)
            is_correct = (preds == tgt).all(dim=-1)
            loss = compute_bce_joint_loss(
                logits, tgt, bce_probs, is_correct.float(), alpha=1.0
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item() * inp.size(0)
            correct += is_correct.sum().item()
            total += inp.size(0)

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for inp, tgt in val_loader:
                inp, tgt = inp.to(device), tgt.to(device)
                logits, _ = model(inp)
                preds = logits.argmax(dim=-1)
                val_correct += (preds == tgt).all(dim=-1).sum().item()
                val_total += inp.size(0)

        print(
            f"Epoch {epoch} | Loss: {total_loss / total:.4f} | Train Acc: {correct / total:.4f} | Val Acc: {val_correct / val_total:.4f}"
        )

    torch.save(
        {"state_dict": model.state_dict(), "config": vars(cfg)},
        f"{DATA_DIR}/sudoku_model.pt",
    )
    return model


sudoku_model = train_sudoku() if RUN_TRAINING else None
if not RUN_TRAINING:
    print("Sudoku training skipped; set RUN_TRAINING = True to enable it.")

### 5b. Maze Solver (10x10)

Trains on small mazes with 2D positional embeddings. Lighter config for quick experimentation.

In [ ]:
def train_maze_small() -> nn.Module | None:
    data_path = f"{DATA_DIR}/maze_dryrun.pt"
    if not os.path.exists(data_path):
        print("Maze dataset not found. Skipping.")
        return None

    raw_samples = torch.load(data_path)
    max_path_len = max(len(sample["path"]) for sample in raw_samples)
    cfg = MambaHybridConfig(
        d_model=64, n_meta=16, l_ans=max_path_len, n_steps=2, t_cycles=2, vocab_size=101
    )
    from scripts.train_maze import MazeDataset, MazeReasoningModel
    from mamba_hybrid.loss import compute_bce_joint_loss

    ds = MazeDataset(data_path, size=10, max_path_len=max_path_len)
    train_size = int(0.8 * len(ds))
    val_size = len(ds) - train_size
    train_set, val_set = torch.utils.data.random_split(ds, [train_size, val_size])
    train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=32, shuffle=False)

    model = MazeReasoningModel(cfg, grid_size=10).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)

    for epoch in range(1, 4):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0
        for grid, tgt in train_loader:
            grid, tgt = grid.to(device), tgt.to(device)
            opt.zero_grad()
            logits, bce_probs = model(grid)
            preds = logits.argmax(dim=-1)
            is_correct = (preds == tgt).all(dim=-1)
            loss = compute_bce_joint_loss(
                logits, tgt, bce_probs, is_correct.float(), alpha=1.0, ignore_index=100
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item() * grid.size(0)
            correct += is_correct.sum().item()
            total += grid.size(0)

        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for grid, tgt in val_loader:
                grid, tgt = grid.to(device), tgt.to(device)
                logits, _ = model(grid)
                preds = logits.argmax(dim=-1)
                val_correct += (preds == tgt).all(dim=-1).sum().item()
                val_total += grid.size(0)

        print(
            f"Epoch {epoch} | Loss: {total_loss / total:.4f} | Train Acc: {correct / total:.4f} | Val Acc: {val_correct / val_total:.4f}"
        )

    torch.save(
        {"state_dict": model.state_dict(), "config": vars(cfg)},
        f"{DATA_DIR}/maze_model.pt",
    )
    return model


maze_model = train_maze_small() if RUN_TRAINING else None
if not RUN_TRAINING:
    print("Maze training skipped; set RUN_TRAINING = True to enable it.")

### 5c. Dijkstra Graph Routing

Trains on 20-node weighted graphs to predict shortest path parent trees.

In [ ]:
def train_dijkstra() -> nn.Module | None:
    data_path = f"{DATA_DIR}/dijkstra.pt"
    if not os.path.exists(data_path):
        print("Dijkstra dataset not found. Skipping.")
        return None

    cfg = MambaHybridConfig(
        d_model=128, n_meta=32, l_ans=20, n_steps=4, t_cycles=3, vocab_size=20
    )
    from scripts.train_dijkstra import DijkstraDataset, DijkstraReasoningModel
    from mamba_hybrid.loss import compute_bce_joint_loss

    all_samples = torch.load(data_path)[:500]
    random.seed(42)
    random.shuffle(all_samples)
    train_size = int(0.8 * len(all_samples))
    train_loader = DataLoader(
        DijkstraDataset(
            all_samples[:train_size], augment=True, num_nodes=20, d_model=128
        ),
        batch_size=16,
        shuffle=True,
    )
    val_loader = DataLoader(
        DijkstraDataset(
            all_samples[train_size:], augment=False, num_nodes=20, d_model=128
        ),
        batch_size=16,
        shuffle=False,
    )

    model = DijkstraReasoningModel(cfg, vocab_size=20).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

    for epoch in range(1, 4):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0
        for feats, tgt in train_loader:
            feats, tgt = feats.to(device), tgt.to(device)
            opt.zero_grad()
            logits, bce_probs = model(feats)
            preds = logits.argmax(dim=-1)
            is_correct = (preds == tgt).all(dim=-1)
            loss = compute_bce_joint_loss(
                logits, tgt, bce_probs, is_correct.float(), alpha=1.0
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item() * feats.size(0)
            correct += is_correct.sum().item()
            total += feats.size(0)

        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for feats, tgt in val_loader:
                feats, tgt = feats.to(device), tgt.to(device)
                logits, _ = model(feats)
                preds = logits.argmax(dim=-1)
                val_correct += (preds == tgt).all(dim=-1).sum().item()
                val_total += feats.size(0)

        print(
            f"Epoch {epoch} | Loss: {total_loss / total:.4f} | Train Acc: {correct / total:.4f} | Val Acc: {val_correct / val_total:.4f}"
        )

    torch.save(
        {"state_dict": model.state_dict(), "config": vars(cfg)},
        f"{DATA_DIR}/dijkstra_model.pt",
    )
    return model


dijkstra_model = train_dijkstra() if RUN_TRAINING else None
if not RUN_TRAINING:
    print("Dijkstra training skipped; set RUN_TRAINING = True to enable it.")

### 5d. Unified Multi-Task (MoE)

Trains a single model on all tasks with task-prefixed routing through Mixture-of-Experts layers.
Each task has its own FFN expert in the hybrid block + its own projection head.

In [ ]:
def train_multitask() -> nn.Module | None:
    from scripts.train_multitask import MultiTaskDataset, UnifiedReasoningLLM

    mt_cfg = MambaHybridConfig(
        d_model=64,
        n_meta=16,
        l_ans=128,
        n_steps=2,
        t_cycles=2,
        use_moe=True,
        vocab_size=128,
    )
    ds = MultiTaskDataset(
        DATA_DIR, max_seq_len=128, l_ans=128, max_samples_per_task=50, required_tasks=()
    )
    if len(ds) == 0:
        print("No multi-task data found. Skipping.")
        return None
    print(f"Loaded {len(ds)} multi-task samples.")

    train_size = int(0.8 * len(ds))
    val_size = len(ds) - train_size
    train_set, val_set = torch.utils.data.random_split(ds, [train_size, val_size])
    train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=16, shuffle=False)

    model = UnifiedReasoningLLM(mt_cfg, vocab_size=128).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)
    from mamba_hybrid.loss import compute_bce_joint_loss

    for epoch in range(1, 3):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0
        for inp_ids, tgt_ids, task_names in train_loader:
            inp_ids, tgt_ids = inp_ids.to(device), tgt_ids.to(device)
            opt.zero_grad()
            logits, bce_probs = model(inp_ids, task_names)
            preds = logits.argmax(dim=-1)
            is_correct = (preds == tgt_ids).all(dim=-1)
            loss = compute_bce_joint_loss(
                logits,
                tgt_ids,
                bce_probs,
                is_correct.float(),
                alpha=1.0,
                ignore_index=0,
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item() * inp_ids.size(0)
            correct += is_correct.sum().item()
            total += inp_ids.size(0)

        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for inp_ids, tgt_ids, task_names in val_loader:
                inp_ids, tgt_ids = inp_ids.to(device), tgt_ids.to(device)
                logits, _ = model(inp_ids, task_names)
                preds = logits.argmax(dim=-1)
                val_correct += (preds == tgt_ids).all(dim=-1).sum().item()
                val_total += inp_ids.size(0)

        print(
            f"Epoch {epoch} | Loss: {total_loss / total:.4f} | Train Acc: {correct / total:.4f} | Val Acc: {val_correct / val_total:.4f}"
        )

    torch.save(
        {"state_dict": model.state_dict(), "config": vars(mt_cfg)},
        f"{DATA_DIR}/unified_model.pt",
    )
    return model


unified_model = train_multitask() if RUN_TRAINING else None
if not RUN_TRAINING:
    print("Multi-task training skipped; set RUN_TRAINING = True to enable it.")

## 6. Loss Functions

### BCE Joint Loss

Combines sparse task cross-entropy (applied only at the final step) with BCE halting supervision.
The halting head learns to predict `correct_mask` (1 if the entire sample is correct, 0 otherwise).

### Q-Learning Joint Loss

Uses bootstrapped Q-learning (SARSA-style) for the halting head, with a target network
providing TD targets. The task CE is still sparse (final step only).

In [ ]:
from mamba_hybrid.loss import compute_bce_joint_loss, compute_q_joint_loss

# BCE loss demo
B, L_ans, V = 4, 81, 10
logits = torch.randn(B, L_ans, V)
target_ids = torch.randint(0, V, (B, L_ans))
bce_probs = [torch.sigmoid(torch.randn(B)) for _ in range(4)]
correct_mask = torch.randint(0, 2, (B,)).float()

loss = compute_bce_joint_loss(logits, target_ids, bce_probs, correct_mask, alpha=1.0)
print(f"BCE Joint Loss: {loss.item():.4f}")

# Q-learning loss demo
cfg = MambaHybridConfig(
    d_model=64, n_meta=16, l_ans=16, n_steps=4, t_cycles=3, vocab_size=V
)
target_model = MambaAttentionHybrid(cfg)

states = [(torch.randn(B, 16, 64), torch.randn(B, 16, 64)) for _ in range(4)]
q_preds = [torch.randn(B, 2) for _ in range(4)]
loss_q = compute_q_joint_loss(
    logits[:, :16],
    target_ids[:, :16],
    q_preds,
    correct_mask,
    target_model,
    alpha=1.0,
    gamma=1.0,
    states=states,
)
print(f"Q Joint Loss: {loss_q.item():.4f}")

## 7. PTRM Inference

**Probabilistic Tiny Recursive Model (PTRM)** performs K stochastic rollouts with Gaussian noise
injection, then selects the consensus answer via majority voting within similarity groups.

This inference-time scaling technique improves accuracy without retraining.

In [ ]:
from mamba_hybrid.inference import ptrm_inference

# Build a sample input for PTRM
cfg_ptrm = MambaHybridConfig(d_model=64, n_meta=16, l_ans=16, n_steps=2, t_cycles=2)
model_ptrm = MambaAttentionHybrid(cfg_ptrm).eval()
X_raw = torch.randn(2, 32, cfg_ptrm.d_model)

# Single deterministic pass (K=1)
y_det = ptrm_inference(X_raw, model_ptrm, K=1)
print(f"Deterministic: {y_det.shape}")

# Stochastic consensus (K=5)
y_stoch = ptrm_inference(X_raw, model_ptrm, K=5, sigma_base=0.05)
print(f"Stochastic consensus: {y_stoch.shape}")

## 8. Evaluation & Visualization

### 8a. Sudoku Board Display

In [ ]:
def print_sudoku(board: List[int]) -> None:
    for r in range(9):
        if r % 3 == 0 and r != 0:
            print("------+-------+------")
        row_str = []
        for c in range(9):
            if c % 3 == 0 and c != 0:
                row_str.append("|")
            row_str.append(str(board[r * 9 + c]) if board[r * 9 + c] != 0 else ".")
        print(" ".join(row_str))


# Quick evaluation on saved model
ckpt_path = f"{DATA_DIR}/sudoku_model.pt"
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    cfg_dict = ckpt.get("config", {})
    eval_cfg = MambaHybridConfig(
        d_model=cfg_dict.get("d_model", 128),
        n_meta=cfg_dict.get("n_meta", 32),
        l_ans=cfg_dict.get("l_ans", 81),
        n_steps=cfg_dict.get("n_steps", 4),
        t_cycles=cfg_dict.get("t_cycles", 3),
        vocab_size=cfg_dict.get("vocab_size", 10),
    )
    from scripts.train_sudoku import SudokuReasoningModel

    eval_model = SudokuReasoningModel(eval_cfg, vocab_size=10).to(device)
    eval_model.load_state_dict(ckpt["state_dict"])
    eval_model.eval()

    all_s = []
    with open(f"{DATA_DIR}/sudoku.jsonl") as f:
        for line in f:
            all_s.append(json.loads(line))
            if len(all_s) >= 5:
                break
    ds = SudokuDataset(all_s, augment=False)
    puzzle, solution = ds[0]

    with torch.no_grad():
        logits, _ = eval_model(puzzle.unsqueeze(0).to(device))
        preds = logits.argmax(dim=-1).squeeze(0).tolist()

    print("=== INPUT PUZZLE ===")
    print_sudoku(puzzle.tolist())
    print("\n=== PREDICTED ===")
    print_sudoku(preds)
    print("\n=== GROUND TRUTH ===")
    print_sudoku(solution.tolist())

    cell_acc = sum(1 for p, t in zip(preds, solution.tolist()) if p == t)
    print(f"\nCell Accuracy: {cell_acc}/81 ({cell_acc / 81 * 100:.1f}%)")
else:
    print("No saved sudoku model found. Train one first in section 5a.")

### 8b. Maze Path Visualization

In [ ]:
def print_maze(grid, true_path=None, pred_path=None):
    size = len(grid)
    chars = [
        ["█" if grid[r][c] == 1 else " " for c in range(size)] for r in range(size)
    ]
    if true_path:
        for r, c in true_path:
            if 0 <= r < size and 0 <= c < size:
                chars[r][c] = "·"
    if pred_path:
        for r, c in pred_path:
            if 0 <= r < size and 0 <= c < size:
                chars[r][c] = "*" if chars[r][c] == "·" else "x"
    chars[0][0] = "S"
    chars[size - 1][size - 1] = "E"
    print("+" + "-" * size + "+")
    for r in range(size):
        print("|" + "".join(chars[r]) + "|")
    print("+" + "-" * size + "+")


maze_ckpt = f"{DATA_DIR}/maze_model.pt"
maze_payload = (
    torch.load(maze_ckpt, map_location=device) if os.path.exists(maze_ckpt) else None
)
if (
    isinstance(maze_payload, dict)
    and "state_dict" in maze_payload
    and os.path.exists(f"{DATA_DIR}/maze_dryrun.pt")
):
    raw_data = torch.load(f"{DATA_DIR}/maze_dryrun.pt")
    max_path_len = max(len(sample["path"]) for sample in raw_data)
    cfg_m = MambaHybridConfig(
        d_model=64, n_meta=16, l_ans=max_path_len, n_steps=2, t_cycles=2, vocab_size=101
    )
    eval_m = MazeReasoningModel(cfg_m, grid_size=10).to(device)
    eval_m.load_state_dict(maze_payload["state_dict"])
    eval_m.eval()

    ds = MazeDataset(f"{DATA_DIR}/maze_dryrun.pt", size=10, max_path_len=max_path_len)
    grid_flat, target_ids = ds[0]

    with torch.no_grad():
        logits, _ = eval_m(grid_flat.unsqueeze(0).to(device))
        preds = logits.argmax(dim=-1).squeeze(0).tolist()

    pred_path = [(t // 10, t % 10) for t in preds if t < 100]
    true_path = [(t // 10, t % 10) for t in target_ids.tolist() if t < 100]

    print("S=Start, E=End, ·=Truth, *=Match, x=Mistake")
    print_maze(raw_data[0]["grid"], true_path, pred_path)
    print(f"True path: {true_path}")
    print(f"Pred path: {pred_path}")
else:
    print("No compatible saved maze model found. Train one in section 5b.")

### 8c. Dijkstra Routing Tree

In [ ]:
dijk_ckpt = f"{DATA_DIR}/dijkstra_model.pt"
if os.path.exists(dijk_ckpt):
    ckpt = torch.load(dijk_ckpt, map_location=device)
    cfg_d = MambaHybridConfig(
        **{
            k: v
            for k, v in ckpt["config"].items()
            if k in ["d_model", "n_meta", "l_ans", "n_steps", "t_cycles"]
        }
    )
    eval_d = DijkstraReasoningModel(cfg_d, vocab_size=20).to(device)
    eval_d.load_state_dict(ckpt["state_dict"])
    eval_d.eval()

    raw_d = torch.load(f"{DATA_DIR}/dijkstra.pt")[0]
    ds = DijkstraDataset([raw_d], augment=False, num_nodes=20, d_model=cfg_d.d_model)
    feats, parents = ds[0]

    with torch.no_grad():
        logits, _ = eval_d(feats.unsqueeze(0).to(device))
        preds = logits.argmax(dim=-1).squeeze(0).tolist()

    print("Node | Predicted Parent | True Parent")
    print("-----+-----------------+------------")
    for i in range(20):
        p_str = "Source" if preds[i] == i else f"Node {preds[i]}"
        t_str = "Source" if parents[i].item() == i else f"Node {parents[i].item()}"
        m = "✓" if preds[i] == parents[i].item() else "✗"
        print(f"{i:4d} | {p_str:15s} | {t_str:10s} {m}")

    correct = sum(1 for p, t in zip(preds, parents.tolist()) if p == t)
    print(f"\nAccuracy: {correct}/20 ({correct / 20 * 100:.1f}%)")
else:
    print("No saved dijkstra model found. Train one first in section 5c.")

### 8d. Multi-Task Evaluation

In [ ]:
def decode_tokens(tokens):
    return "".join(chr(t) for t in tokens if t != 0)


uni_ckpt = f"{DATA_DIR}/unified_model.pt"
ckpt = torch.load(uni_ckpt, map_location=device) if os.path.exists(uni_ckpt) else None
if isinstance(ckpt, dict) and "config" in ckpt and "state_dict" in ckpt:
    cfg_dict = ckpt["config"]
    sig = __import__("inspect").signature(MambaHybridConfig.__init__)
    valid = {k for k in sig.parameters if k != "self"}
    filtered = {k: v for k, v in cfg_dict.items() if k in valid}
    cfg_u = MambaHybridConfig(**filtered)
    eval_u = UnifiedReasoningLLM(cfg_u, vocab_size=128).to(device)
    eval_u.load_state_dict(ckpt["state_dict"])
    eval_u.eval()

    ds = MultiTaskDataset(
        DATA_DIR, max_seq_len=128, l_ans=cfg_u.l_ans, max_samples_per_task=5
    )
    task_samples = {"MAZE": [], "SUDOKU": [], "DIJKSTRA": [], "GSM8K": []}
    for i in range(len(ds)):
        inp, tgt, tn = ds[i]
        if tn in task_samples:
            task_samples[tn].append((inp, tgt, tn))

    for tn, samples in task_samples.items():
        if not samples:
            continue
        inp, tgt, _ = samples[0]
        with torch.no_grad():
            logits, _ = eval_u(inp.unsqueeze(0).to(device), [tn])
            preds = logits.argmax(dim=-1).squeeze(0).tolist()
        print(f"\n=== {tn} ===")
        print(f"Input:    {decode_tokens(inp.tolist())[:80]}")
        print(f"Expected: {decode_tokens(tgt.tolist())[:80]}")
        print(f"Predicted:{decode_tokens(preds)[:80]}")
else:
    print("No compatible saved unified model found. Train one in section 5d.")

## 9. Model Architecture Summary

```
Input (X_raw) [B, L_raw, D]
  │
  ├── init_answer() → pooled projection → y [B, L_ans, D]
  ├── M_meta (learned) → z [B, N_meta, D]
  │
  ├── Initial refinement (T-1 cycles, gradients retained)
  │     └── for each cycle:
  │           ├── n_steps of: z = HybridBlock([z, y, X_raw])[:, :N_meta]
  │           └── AnswerUpdateBlock: y = CrossAttn(z, y)
  │
  └── Supervised Phase (T cycle, grad enabled)
        ├── for each step:
        │     ├── z = HybridBlock([z, y, X_raw])[:, :N_meta]
        │     └── ACT Halting: bce_prob (or q_vals)
        └── AnswerUpdateBlock: y = CrossAttn(z, y)
              └── y_final → token_generator → logits
```

### Hybrid Block Detail
```
Input x
  │
  ├── in_proj → split → [q,k,v] + [x_ssm, gate] + [h_in, h_out, delta]
  │
  ├── Attention Branch          ├── SSM Branch
  │     q,k,v → PrefixCausal    │     x_ssm, h_in/h_out, delta
  │            Attention         │         → Mamba2 SSD Scan
  │     → y_attn                │     → y_ssm
  │
  ├── RMSNorm(y_attn)*β₁ + RMSNorm(y_ssm)*β₂
  ├── /2 → y_fused
  ├── [Optional: MoE FFN per-task expert]
  └── out_proj → residual: x + out_proj(y_fused)
```

## 10. Key Design Principles

1. **Sparse Final-Step Supervision** — Loss is applied only to the final answer state of the
   supervised cycle, preventing shortcut memorization of latent sequence replay.

2. **Full-Recursion BPTT** — Gradients flow through the full computational graph of the supervision
   cycle. No IFT gradient approximation is used.

3. **Detached Pooling in ACT Head** — The halting head uses `.detach()` on the pooled state,
   preventing gradients from the halting objective from corrupting the planning representations.

4. **Full-Cycle Gradients** — All refinement cycles retain their computational graphs, so
   supervision updates the learned initial states through the complete recursion.

5. **Binary Reward for ACT** — Reward is 1 if the full answer is correct, 0 otherwise.
   No handcrafted reward shaping or negative step penalties.

6. **PTRM Consensus** — Stochastic noise injection during inference with K rollouts and
   majority voting improves accuracy without retraining.

## 11. Training Commands (CLI Reference)

| Task | Command |
|------|---------|
| Sudoku | `uv run python -m scripts.train_sudoku` |
| Maze 10×10 | `uv run python -m scripts.train_maze` |
| Maze 30×30 | `uv run python -m scripts.train_maze_laptop` |
| Dijkstra | `uv run python -m scripts.train_dijkstra` |
| Multi-Task | `uv run python -m scripts.train_multitask` |
| Data Gen | `uv run python -m scripts.download_all_datasets` |

Use `nohup` + `&` on remote servers to run in background: `nohup uv run python -m scripts.train_sudoku > sudoku.log 2>&1 &`